# TF-IDF Baseline
**Project:** Origin | **Status:** starter template — not yet implemented

## Objective
Fit the lexical vectorizer on project records and inspect Top-K retrieval.

**Expected output:** `results/tfidf_*.csv`

> Research decisions in the report are provisional. Do not claim evaluated performance before running experiments.


## 1. Setup and dependencies
Use Google Colab and authenticate to the repository securely. Each notebook must load its own inputs. Never print secrets.


In [15]:
%pip -q install pandas scikit-learn openpyxl

In [16]:
import io, json, re
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

OUTPUT_DIR = Path('results')
OUTPUT_DIR.mkdir(exist_ok=True)
TOP_K = 5
RANDOM_STATE = 42
print('Ready. scikit-learn version loaded.')

Ready. scikit-learn version loaded.


## 2. Load inputs and verify prerequisites
Document file paths, data assumptions, and expected schema before running experiments.


In [17]:
from google.colab import files
uploaded = files.upload()
assert uploaded, 'Please upload the validated project dataset.'
filename = next(iter(uploaded))
contents = io.BytesIO(uploaded[filename])
if filename.lower().endswith('.csv'):
    try:
        df = pd.read_csv(contents, encoding='utf-8-sig')
    except UnicodeDecodeError:
        contents.seek(0)
        df = pd.read_csv(contents, encoding='cp1256')
elif filename.lower().endswith(('.xlsx','.xls')):
    df = pd.read_excel(contents)
else:
    raise ValueError('Upload a CSV or Excel dataset.')
print('Dataset shape:', df.shape)
print('Available columns:', df.columns.tolist())
display(df.head(3))

Saving projects_clean (1).csv to projects_clean (1).csv
Dataset shape: (103, 7)
Available columns: ['Project_ID', 'Title', 'Abstract', 'Department', 'Year', 'Keywords', 'Source_Page']


,Project_ID,Title,Abstract,Department,Year,Keywords,Source_Page
0,P001,Automatic Plant Watering and Tracking System,Many people struggle with watering their plant...,Computer Science,2024 – 2025,"Moisturem Air humidity, AWTS system, Waterin, ...",5
1,P002,علامة CourseAssist: CLO Performance Assistant ...,The rapid evolution of educational technology ...,Computer Science,2024 – 2025,"Program Learning Outcomes, Course Learning Out...",6
2,P003,واصف Mobile Application for Deaf and Mute indi...,"Our System, Wasif, offers a communication appl...",Computer Science,2024 – 2025,"Arabic Sign Language, Sign Language Gestures i...",7


In [18]:
# Adjust these names ONLY if auto-detection does not match your dataset.
OVERRIDE_ID = None
OVERRIDE_TITLE = None
OVERRIDE_DESCRIPTION = None
OVERRIDE_TEXT = None  # Use only when one column already contains complete project text

aliases = {
    'id': ['project_id', 'id', 'project number', 'project_number', 'رقم المشروع', 'معرف المشروع'],
    'title': ['title', 'project_title', 'project name', 'project_name', 'اسم المشروع', 'عنوان المشروع'],
    'description': ['description', 'abstract', 'project_description', 'summary', 'idea', 'الوصف', 'وصف المشروع', 'ملخص', 'الفكرة'],
    'text': ['combined_text', 'project_text', 'cleaned_text', 'text', 'full_text', 'search_text']
}

def detect_column(columns, candidates):
    lookup = {str(c).strip().casefold(): c for c in columns}
    return next((lookup[a.casefold()] for a in candidates if a.casefold() in lookup), None)

id_col = OVERRIDE_ID or detect_column(df.columns, aliases['id'])
title_col = OVERRIDE_TITLE or detect_column(df.columns, aliases['title'])
desc_col = OVERRIDE_DESCRIPTION or detect_column(df.columns, aliases['description'])
text_col = OVERRIDE_TEXT or detect_column(df.columns, aliases['text'])
print({'id': id_col, 'title': title_col, 'description': desc_col, 'text': text_col})
assert text_col is not None or title_col is not None or desc_col is not None, (
    'Could not identify text columns. Set OVERRIDE_TITLE / OVERRIDE_DESCRIPTION / OVERRIDE_TEXT above.'
)
assert df.shape[0] > 0, 'No rows in dataset.'

{'id': 'Project_ID', 'title': 'Title', 'description': 'Abstract', 'text': None}


In [19]:
def clean_text(value):
    # Minimal normalization only: keep Arabic and English letters and avoid aggressive stemming.
    if pd.isna(value):
        return ''
    txt = str(value).strip().lower()
    txt = re.sub(r'[\u064b-\u065f\u0670]', '', txt)  # Arabic diacritics
    txt = re.sub(r'\s+', ' ', txt)
    return txt

records = pd.DataFrame()
records['project_id'] = (df[id_col].astype(str) if id_col else pd.Series(
    [f'P{i:03d}' for i in range(1, len(df)+1)], index=df.index))
records['title'] = df[title_col].fillna('').astype(str) if title_col else ''
records['description'] = df[desc_col].fillna('').astype(str) if desc_col else ''
if text_col is not None:
    records['search_text'] = df[text_col].map(clean_text)
else:
    records['search_text'] = (records['title'] + ' ' + records['description']).map(clean_text)
records = records[records['search_text'].str.len() > 0].copy().reset_index(drop=True)
assert len(records) > 0, 'No nonempty searchable texts.'
assert records['project_id'].is_unique, 'Project IDs are not unique: check source data or ID selection.'
print('Valid project records:', len(records))
print('Empty records skipped:', len(df) - len(records))
print('Repeated text rows:', records['search_text'].duplicated().sum())
display(records[['project_id','title','search_text']].head(3))

Valid project records: 103
Empty records skipped: 0
Repeated text rows: 0


,project_id,title,search_text
0,P001,Automatic Plant Watering and Tracking System,automatic plant watering and tracking system m...
1,P002,علامة CourseAssist: CLO Performance Assistant ...,علامة courseassist: clo performance assistant ...
2,P003,واصف Mobile Application for Deaf and Mute indi...,واصف mobile application for deaf and mute indi...


## 3. Implementation / experiments
Build and test this step incrementally; document any methodological changes.


In [20]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    analyzer='word',
    ngram_range=(1, 2),
    min_df=1,
    max_df=1.0,
    sublinear_tf=True,
    token_pattern=r'(?u)\b\w+\b'
)
corpus_matrix = vectorizer.fit_transform(records['search_text'])
print('TF-IDF document-term matrix:', corpus_matrix.shape)
print('Vocabulary size:', len(vectorizer.vocabulary_))

TF-IDF document-term matrix: (103, 16881)
Vocabulary size: 16881


In [21]:
def retrieve(query, k=TOP_K, excluded_project_id=None):
    query = clean_text(query)
    if not query:
        raise ValueError('Query must not be empty.')
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, corpus_matrix).ravel()
    candidates = np.argsort(-sims, kind='stable')
    if excluded_project_id is not None:
        candidates = [i for i in candidates if records.iloc[i]['project_id'] != str(excluded_project_id)]
    selected = list(candidates)[:min(k, len(candidates))]
    ans = records.iloc[selected][['project_id','title','description']].copy()
    ans.insert(0, 'rank', range(1, len(ans)+1))
    ans['cosine_similarity'] = np.round(sims[selected], 6)
    return ans.reset_index(drop=True)

# Test with a NEW project idea rather than a corpus record.
TEST_QUERY = 'AI system for recognizing Arabic sign language and helping deaf people communicate'
result = retrieve(TEST_QUERY, k=TOP_K)
display(result)

,rank,project_id,title,description,cosine_similarity
0,1,P069,Bridging Communication Gaps: A system for Arab...,Sign language plays a critical role in communi...,0.175714
1,2,P003,واصف Mobile Application for Deaf and Mute indi...,"Our System, Wasif, offers a communication appl...",0.119429
2,3,P036,رَفد RafdSign: Audio-to-Sign Language Translator,Individuals with hearing impairments face sign...,0.109432
3,4,P047,ASMAAK | أسمعك,The “ASMAAK | أسمعك” project is a child-friend...,0.087267
4,5,P074,بصيرة Baseera: An AI-Based Sound Recognition a...,Individuals with hearing impairments often enc...,0.051667


## 4. Validate results
Run sanity checks, inspect failure cases, and avoid interpreting uncalibrated scores as duplication probabilities.


In [22]:
# Verify rankings, ranges, and query vocabulary coverage.
assert len(result) <= TOP_K
assert result['cosine_similarity'].between(0,1).all()
assert result['cosine_similarity'].is_monotonic_decreasing
assert result['project_id'].is_unique

query_vec = vectorizer.transform([clean_text(TEST_QUERY)])
query_known_terms = query_vec.nnz
print('TEST_QUERY:', TEST_QUERY)
print('Known query terms/ngrams:', query_known_terms)
print('Top similarity:', result['cosine_similarity'].iloc[0] if len(result) else 'N/A')
if query_known_terms == 0:
    print('WARNING: Query shares no known vocabulary with this corpus; all scores can be zero.')
print('Checks passed. Retrieval relevance NOT YET EVALUATED.')

TEST_QUERY: AI system for recognizing Arabic sign language and helping deaf people communicate
Known query terms/ngrams: 17
Top similarity: 0.175714
Checks passed. Retrieval relevance NOT YET EVALUATED.


## 5. Save outputs and conclusions
Record output paths, reproducibility notes, decisions, and next actions.


In [23]:
topk_export = result.copy()
topk_export.insert(0, 'query_id', 'Q_DEMO_001')
topk_export.insert(1, 'query_text', TEST_QUERY)
# Exclude potentially restricted descriptions from shareable outputs; check permissions before committing.
topk_export = topk_export.drop(columns=['description'])
topk_export.to_csv(OUTPUT_DIR/'tfidf_topk.csv', index=False, encoding='utf-8-sig')
pd.DataFrame([{'query_id':'Q_DEMO_001', 'query_text':TEST_QUERY,
               'type':'demo_only_not_human_labeled'}]).to_csv(
    OUTPUT_DIR/'tfidf_queries.csv', index=False, encoding='utf-8-sig')
summary = {
    'model':'TfidfVectorizer + cosine_similarity',
    'ngram_range':[1,2], 'min_df':1, 'max_df':1.0,
    'sublinear_tf':True, 'top_k':TOP_K,
    'source_file':filename, 'input_row_count':len(df), 'valid_record_count':len(records),
    'vocabulary_size':len(vectorizer.vocabulary_),
    'evaluation_status':'Not evaluated against human relevance judgments',
    'notes':'Scores are cosine similarities, not probability of duplication.'
}
(OUTPUT_DIR/'tfidf_run_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved files:')
for path in OUTPUT_DIR.glob('tfidf_*'):
    print(path, '|', path.stat().st_size, 'bytes')

Saved files:
results/tfidf_topk.csv | 915 bytes
results/tfidf_queries.csv | 150 bytes
results/tfidf_run_summary.json | 437 bytes


In [24]:
# Optional: Download the three result files in one ZIP from Colab.
import shutil
from google.colab import files
archive = shutil.make_archive('origin_tfidf_results', 'zip', root_dir='.', base_dir='results')
files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>